# Notebook_C (dùng chung, phần dữ liệu, Sinh viên A): thu thập, so sánh nguồn, làm sạch, EDA và SQL

**Bản đã điền sẵn cho đề tài USGS**: dự báo nhiệt độ nước sông 1 và 3 ngày tại các trạm liên tục USGS (NWIS) từ lưu lượng, oxy hoà tan, độ đục và thời tiết GHCN. Nguồn chính và nguồn phụ đều đọc từ **CSV trong `data/raw`**; ô "Bước 1-prep" tạo `data/raw/primary.csv` từ cache NWIS của Notebook_A (hoặc tải mới nếu thiếu).

**Điểm khác so với template chung (đều có chú thích trong ô code):**

* Lưới thời gian ngày được giữ **đầy đủ** (`clean_grid`): LAG, LEAD và cửa sổ trượt chạy trên lưới này nên "1 ngày", "7 ngày" luôn là ngày lịch, không bị lệch khi trạm thiếu dữ liệu. Bảng `feat` chỉ giữ dòng có mục tiêu sau khi đã tính cửa sổ.
* Đặc trưng **naive mùa đúng tầm**: với tầm h và chu kỳ `SEASON`, dự báo naive mùa là giá trị cách `SEASON - h` ngày trước thời điểm dự báo (`y_lag{SEASON-h}`), tức đúng giá trị một mùa trước thời điểm đích.
* Thêm đặc trưng chu kỳ năm `doy_sin`, `doy_cos` (SEASON = 7 chỉ bắt chu kỳ tuần; chu kỳ năm đi qua `doy`, `mon`).
* Đổi ngưỡng giữ trạm từ "90% của trạm dài nhất" sang `MIN_UNIT_STEPS` ngày có mục tiêu (mặc định 500, giống Notebook_A) để không loại gần hết các trạm ngắn.
* Thêm truy vấn Q5 (độ đục sau mưa) và Q6 (ngày oxy thấp) cho RQ1 của đề cương.

**Ba câu hỏi nghiên cứu chung (áp dụng cho mọi đề, mỗi nhóm thêm 1-2 câu riêng theo đề):**

* **RQ1 (dữ liệu và mô tả, sâu hơn)**: Hai nguồn dữ liệu (chính và phụ) có nhất quán với nhau không, biến mục tiêu biến động theo mùa, theo đơn vị và theo biến phụ như thế nào, và phần nào của biến động được giải thích bằng lịch so với bằng nguồn phụ?
* **RQ2 (mô hình và tổng quát hoá)**: Trong sáu họ mô hình (mốc naive, ridge, random forest, LightGBM toàn cục, tuyến tính dài hạn, mô hình nền zero-shot), mô hình nào tốt nhất theo tầm dự báo, và kết quả có giữ được khi lùi gốc dự báo theo thời gian và khi gặp đơn vị chưa thấy không?
* **RQ3 (tin cậy và vận hành)**: Khoảng tin cậy có đúng bao phủ không, cảnh báo sự kiện (nhiệt độ nước vượt 20 C) đạt recall và precision bao nhiêu, sai số thay đổi thế nào khi chế độ đổi, và nguồn phụ đóng góp bao nhiêu (ablation)?

Notebook_C trả lời RQ1 và chuẩn bị dữ liệu cho RQ2, RQ3 (Notebook_D).

In [1]:
# CONFIG for the USGS water quality topic (Notebook_D reads the same values from manifest.json)
TOPIC = "Three-day river water temperature forecasting at continuous USGS stations from discharge and weather"
GROUP = "Nhom 12"                    # group label used in file names and the audit log
PRIMARY_SOURCE = {"name": "USGS NWIS daily values (dataretrieval)", "url": "https://waterservices.usgs.gov/nwis/dv", "license": "US public domain", "path": "data/raw/primary.csv"}
SECOND_SOURCE  = {"name": "NOAA GHCN-Daily, nearest station (PRCP, TMAX)", "url": "https://www.ncei.noaa.gov/pub/data/ghcn/daily/", "license": "NOAA open data", "path": "data/raw/ghcn_wq_daily.csv"}
UNIT_COL   = "station"               # USGS site number (string, keep leading zeros)
TIME_COL   = "date"                  # daily timestamp
TARGET_COL = "wtemp"                 # water temperature, degrees C (parameter 00010)
EXOG_COLS  = ["TMAX", "PRCP"]        # covariates from the secondary source (air temperature, precipitation)
PRIM_COVS  = ["dox", "turb", "flow"] # other NWIS parameters of the same station: dissolved oxygen, turbidity, discharge
FREQ       = "D"                     # daily grid
HORIZONS   = [1, 3]                  # forecast horizons in days
H_MAIN     = 3                       # headline horizon of the paper (three-day forecast)
SEASON     = 7                       # lag/rolling window in days; the annual cycle is carried by doy, mon, doy_sin, doy_cos
TEST_START = "2023-01-01"            # time-based split: train before, test from this date
EVENT_ABS  = 20.0                    # thermal event = water temperature above 20 C (salmonid stress); set None to use EVENT_QUANTILE
EVENT_QUANTILE = 0.9                 # used only when EVENT_ABS is None
MIN_UNIT_STEPS = 500                 # keep stations with at least this many days that have a target (same rule as Notebook_A)
PHYSICAL_RANGES = {"wtemp": (-2, 40), "dox": (0, 20), "turb": (0, 5000), "flow": (0, 1e6)}   # physical ranges, out-of-range values become NaN
print("Topic:", TOPIC); print("Group:", GROUP)

Topic: Three-day river water temperature forecasting at continuous USGS stations from discharge and weather
Group: Nhom 12


In [2]:
# AI Audit Log helper: every prompt that changed your work is one row (2 minutes per entry, 3-5 per week)
import pandas as pd, os, datetime as dt
os.makedirs("report", exist_ok=True)
AUDIT_PATH = "report/ai_audit_log.csv"
def audit(step, prompt, tool, ai_output_summary, verified_how, decision, hallucination=False):
    """Append one entry. decision: what you kept / changed / rejected. hallucination=True if the AI answer was wrong and you caught it."""
    row = {"date": dt.date.today().isoformat(), "group": GROUP, "step": step, "prompt": prompt[:500], "tool": tool,
           "ai_output": ai_output_summary[:500], "verified_how": verified_how[:300], "decision": decision[:300], "hallucination": int(hallucination)}
    df = pd.DataFrame([row])
    df.to_csv(AUDIT_PATH, mode="a", header=not os.path.exists(AUDIT_PATH), index=False)
    print("audit entry saved:", step)
# example (delete after reading): audit("Step 2", "Given columns ... how to treat gaps longer than 3 steps?", "Claude", "suggested interpolate(limit=3) then drop", "checked share of gaps > 3 in Q4 below", "kept limit=3, dropped 1.2% rows")

## Hướng dẫn hỏi AI (điền đề tài của nhóm vào chỗ trống)

Quy tắc: hỏi **cụ thể** (kèm tên cột, kích thước, thông báo lỗi), yêu cầu AI **giải thích lý do** và **nêu cách kiểm tra**, rồi tự kiểm tra trước khi dùng. Mỗi prompt làm thay đổi bài phải ghi vào AI Audit Log bằng hàm `audit(...)` ở ô trên. Mẫu prompt theo bước (thay `{TOPIC}`, `{cột}` bằng thông tin thật của nhóm):

| Bước | Mẫu prompt | Cần tự kiểm tra gì |
|---|---|---|
| Thu thập | "Đề tài của tôi là {TOPIC}. Nguồn chính là {PRIMARY_SOURCE}. Hãy gợi ý 3 nguồn phụ công khai (thời tiết, sự kiện, giá, lịch) có thể ghép theo cột {TIME_COL} và {UNIT_COL}, kèm URL tải và giấy phép." | Mở URL, xác nhận có tải được và giấy phép cho phép dùng |
| So sánh nguồn | "Tôi có hai nguồn về cùng biến {TARGET_COL} ở tần suất {FREQ}. Hãy đề xuất 4 chỉ số để so sánh độ phủ, độ trễ cập nhật và độ lệch giữa hai nguồn." | Chạy lại số trên dữ liệu thật, không dùng số AI đưa |
| Làm sạch | "Cột {cột} có {x}% thiếu, thiếu theo cụm dài nhất {n} bước. Nên nội suy hay bỏ? Giải thích rủi ro rò rỉ tương lai." | Kiểm tra nội suy không dùng giá trị tương lai |
| SQL | "Viết truy vấn DuckDB tạo đặc trưng trễ {SEASON} bước và trung bình trượt theo {UNIT_COL}, có WINDOW, không rò rỉ." | Đếm dòng, kiểm tra cột NULL ở đầu chuỗi |
| EDA | "Từ bảng thống kê sau (dán bảng), nêu 3 nhận xét có thể kiểm chứng và 2 điều cần cảnh giác." | Mỗi nhận xét phải chỉ ra được ô số liệu tương ứng |
| Lỗi | "Lỗi: {dán nguyên văn}. Ngữ cảnh: {ô code}. Nguyên nhân và cách sửa tối thiểu?" | Sửa xong chạy lại ô test |

Ví dụ ghi Audit Log sau khi hỏi: `audit("Thu thập", "Đề tài ... gợi ý nguồn phụ", "Claude", "3 nguồn: NOAA ISD, ...", "mở 3 URL, 1 URL lỗi 404", "dùng NOAA ISD, loại nguồn 404", hallucination=True)`.

## Bước 0: môi trường và cấu trúc thư mục

**Làm gì**: tạo môi trường ảo một lần, cài thư viện, tạo thư mục chuẩn. **Vì sao**: cả hai sinh viên và giảng viên chạy cùng phiên bản thư viện thì kết quả tái tạo được; `requirements.txt` là bằng chứng.

**Cấu trúc**: `data/raw` (file gốc, không sửa), `data/processed` (parquet sạch), `sql/` (truy vấn), `report/` (bảng, hình, nhật ký), `notebooks/`.

In [3]:
# Step 0a: run once in a terminal (not in the notebook)
# python -m venv .venv && source .venv/bin/activate      (Windows: .venv\Scripts\activate)
# pip install pandas numpy duckdb pyarrow scikit-learn lightgbm statsmodels matplotlib seaborn ydata-profiling mapie shap requests dataretrieval scipy
# pip freeze > requirements.txt
import os
for d in ["data/raw", "data/processed", "sql", "report", "notebooks"]: os.makedirs(d, exist_ok=True)
print("folders ready")

folders ready


In [4]:
# Step 0b: imports and plotting style (English labels in figures, no top/right spines, no in-figure titles)
import pandas as pd, numpy as np, duckdb, json, hashlib, warnings, re
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore"); np.random.seed(42)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"font.family": "DejaVu Serif", "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
TEAL, ACC, GREY = "#1B6B6D", "#F4A261", "#9FBFBF"
def style(ax): ax.spines[["top", "right"]].set_visible(False)
def savefig(fig, name): fig.tight_layout(); fig.savefig(f"report/{name}.png", dpi=300); plt.close(fig); print("saved report/" + name + ".png")
con = duckdb.connect()
print("pandas", pd.__version__, "duckdb", duckdb.__version__)

pandas 2.3.3 duckdb 1.5.5


## Bước 1: bài toán, nguồn dữ liệu và kế hoạch thu thập

**Làm gì**: viết vấn đề bằng 5 câu, đăng ký hai nguồn (chính và phụ) vào sổ nguồn (`report/data_sources.csv`) với URL, giấy phép, ngày tải, số dòng. **Vì sao**: phần Data của bài báo cần ghi rõ nguồn là công trình nào, năm nào, số dòng, số cột và lý do chọn; sổ nguồn giúp viết phần đó trong hai phút.

**Nguồn phụ nên chọn**: biến có cơ chế ảnh hưởng rõ đến mục tiêu (thời tiết với nhu cầu, giá với sạc, sự kiện với lưu lượng), có cùng hoặc cao hơn tần suất mục tiêu, và có thể ghép theo thời gian và đơn vị.

### Bước 1-prep: tạo `data/raw/primary.csv` (nguồn chính NWIS) và kiểm tra nguồn phụ GHCN

**Làm gì**: ghép mọi file `data/raw/_cache/dv_<site>.parquet` do Notebook_A tải về thành một CSV dài (`station, date, wtemp, dox, turb, flow`). Nếu thiếu cache thì tải lại từ NWIS (có retry). Nguồn phụ `data/raw/ghcn_wq_daily.csv` (cột `station, date, PRCP, TMAX`) do Notebook_A bước 2c tạo; danh sách trạm lấy theo file này.

**Vì sao**: C đọc dữ liệu từ CSV thay vì gọi API, nên hai sinh viên và giảng viên chạy lại được mà không phụ thuộc server USGS (từng trả 503). Mã lỗi cơ quan chỉ gồm các mã âm và 999999; không thay 999 hoặc 9999 vì đó có thể là lưu lượng thật.

**Kiểm tra**: số trạm trong `primary.csv` bằng số trạm trong file GHCN; `wtemp` có giá trị ở hầu hết các trạm.

In [5]:
# Step 1-prep: build data/raw/primary.csv from the NWIS cache of Notebook_A (or download), and check the GHCN CSV
import glob, time, pathlib
PARAM_MAP = {"wtemp": "00010", "dox": "00300", "turb": "63680", "flow": "00060"}
AGENCY_MISSING = [-999999, -99999, -9999, -999, 999999]          # agency "no data" codes (positive 999 and 9999 can be real discharge, so they are kept)
START, END = "2018-01-01", "2026-09-14"
CACHE = pathlib.Path("data/raw/_cache")

assert os.path.exists(SECOND_SOURCE["path"]), f"{SECOND_SOURCE['path']} not found: run Notebook_A steps 2a-2c first (it writes this GHCN file)"
sec_ids = pd.read_csv(SECOND_SOURCE["path"], dtype={UNIT_COL: str}, usecols=[UNIT_COL])[UNIT_COL].unique().tolist()

def coalesce_param(d, code):
    """One value per day for a parameter: Mean sensor first, then (Max+Min)/2, then Median (same priority as Notebook_A)."""
    out = pd.Series(np.nan, index=d.index)
    for c in [c for c in d.columns if c.startswith(code) and c.endswith("_Mean")]: out = out.fillna(d[c])
    mx = [c for c in d.columns if c.startswith(code) and c.endswith("_Maximum")]
    mn = [c for c in d.columns if c.startswith(code) and c.endswith("_Minimum")]
    for b in set(c.rsplit("_Maximum", 1)[0] for c in mx) & set(c.rsplit("_Minimum", 1)[0] for c in mn):
        out = out.fillna((d[b + "_Maximum"] + d[b + "_Minimum"]) / 2)
    for c in [c for c in d.columns if c.startswith(code) and c.endswith("_Median")]: out = out.fillna(d[c])
    return out

def tidy_dv(d, site):
    d = d.copy()
    if "datetime" in d.columns: d = d.rename(columns={"datetime": "date"})
    elif "date" not in d.columns and "index" in d.columns: d = d.rename(columns={"index": "date"})
    d["date"] = pd.to_datetime(d["date"]).dt.tz_localize(None).dt.normalize()
    for c in d.select_dtypes("number").columns: d.loc[d[c].isin(AGENCY_MISSING), c] = np.nan
    out = pd.DataFrame({UNIT_COL: site, TIME_COL: d["date"].values})
    for k, code in PARAM_MAP.items(): out[k] = coalesce_param(d, code).values
    return out

def fetch_dv(site):
    f = CACHE / f"dv_{site}.parquet"
    if f.exists(): return pd.read_parquet(f)
    from dataretrieval import nwis
    for k in range(5):
        try:
            res = nwis.get_dv(sites=[site], parameterCd=list(PARAM_MAP.values()), start=START, end=END)
            d = res[0].reset_index(); d["site_no"] = site; CACHE.mkdir(parents=True, exist_ok=True); d.to_parquet(f, index=False); return d
        except Exception as e:
            print(f"  {site} try {k + 1}/5 failed: {str(e)[:60]}"); time.sleep(3 * 2 ** k)
    return pd.DataFrame()

if not os.path.exists(PRIMARY_SOURCE["path"]):
    parts, missing = [], []
    for s in sec_ids:
        d = fetch_dv(s)
        if d is None or d.empty: missing.append(s); continue
        parts.append(tidy_dv(d, s))
    prim = pd.concat(parts, ignore_index=True).dropna(subset=list(PARAM_MAP), how="all")
    prim = prim[(prim[TIME_COL] >= START) & (prim[TIME_COL] <= END)]
    prim.to_csv(PRIMARY_SOURCE["path"], index=False)
    print("wrote", PRIMARY_SOURCE["path"], prim.shape, "| stations without NWIS data:", missing)
else:
    print(PRIMARY_SOURCE["path"], "already exists, skipped (delete it to rebuild)")
print("stations in the GHCN file:", len(sec_ids))

wrote data/raw/primary.csv (99995, 6) | stations without NWIS data: []
stations in the GHCN file: 37


In [6]:
# Step 1a: source registry (append every file you download; this becomes Table "Data sources" in the paper)
SOURCES_PATH = "report/data_sources.csv"
def register_source(src, rows=None, cols=None, start=None, end=None, note=""):
    """src: dict with name, url, license, path. rows/cols/start/end are filled after loading."""
    row = {**src, "downloaded": pd.Timestamp.today().date().isoformat(), "rows": rows, "cols": cols, "start": start, "end": end, "note": note}
    df = pd.DataFrame([row]); df.to_csv(SOURCES_PATH, mode="a", header=not os.path.exists(SOURCES_PATH), index=False)
    return row
print("registry at", SOURCES_PATH)

registry at report/data_sources.csv


In [7]:
# Step 1b: load the primary source from CSV (unit ids stay strings so the leading zeros of USGS site numbers survive)
def load_table(path):
    """Read csv or parquet into a pandas DataFrame."""
    if path.endswith(".parquet"): return pd.read_parquet(path)
    return pd.read_csv(path, dtype={UNIT_COL: str}, low_memory=False)
raw = load_table(PRIMARY_SOURCE["path"])
raw.columns = [c.strip() for c in raw.columns]
raw[TIME_COL] = pd.to_datetime(raw[TIME_COL], errors="coerce", utc=False)
print(raw.shape); display(raw.head(3)); display(raw.dtypes.value_counts())
register_source(PRIMARY_SOURCE, rows=len(raw), cols=raw.shape[1], start=str(raw[TIME_COL].min()), end=str(raw[TIME_COL].max()))

(99995, 6)


,station,date,wtemp,dox,turb,flow
0,01438500,2018-01-01,NaN,NaN,NaN,2540.0
1,01438500,2018-01-02,NaN,NaN,NaN,2740.0
2,01438500,2018-01-03,NaN,NaN,NaN,2850.0


float64           4
object            1
datetime64[ns]    1
Name: count, dtype: int64

{'name': 'USGS NWIS daily values (dataretrieval)',
 'url': 'https://waterservices.usgs.gov/nwis/dv',
 'license': 'US public domain',
 'path': 'data/raw/primary.csv',
 'downloaded': '2026-10-04',
 'rows': 99995,
 'cols': 6,
 'start': '2018-01-01 00:00:00',
 'end': '2026-09-14 00:00:00',
 'note': ''}

In [8]:
# Step 1c: load the secondary source (GHCN covariates); it shares TIME_COL and UNIT_COL (every USGS station has its own nearest weather station)
sec = load_table(SECOND_SOURCE["path"]); sec.columns = [c.strip() for c in sec.columns]
sec[TIME_COL] = pd.to_datetime(sec[TIME_COL], errors="coerce")
SEC_HAS_UNIT = UNIT_COL in sec.columns
print(sec.shape, "unit-specific:", SEC_HAS_UNIT); display(sec.head(3))
register_source(SECOND_SOURCE, rows=len(sec), cols=sec.shape[1], start=str(sec[TIME_COL].min()), end=str(sec[TIME_COL].max()))

(114219, 4) unit-specific: True


,station,date,PRCP,TMAX
0,01438500,2018-01-01,0.0,-12.8
1,01438500,2018-01-02,0.0,-8.9
2,01438500,2018-01-03,0.0,-4.4


{'name': 'NOAA GHCN-Daily, nearest station (PRCP, TMAX)',
 'url': 'https://www.ncei.noaa.gov/pub/data/ghcn/daily/',
 'license': 'NOAA open data',
 'path': 'data/raw/ghcn_wq_daily.csv',
 'downloaded': '2026-10-04',
 'rows': 114219,
 'cols': 4,
 'start': '2018-01-01 00:00:00',
 'end': '2026-09-14 00:00:00',
 'note': ''}

### Xuất và đọc lại dữ liệu bằng CSV

**Quy ước file**: mọi bước lưu một file CSV riêng để giảng viên và bạn cùng nhóm mở được bằng Excel mà không cần Python: `data/raw/primary_snapshot.csv` (bản gốc đã chuẩn tên cột và kiểu thời gian, không sửa giá trị), `data/processed/clean.csv` (sau làm sạch, một dòng một đơn vị-thời điểm), `data/processed/feat.csv` (bảng đặc trưng bàn giao). Parquet vẫn lưu song song vì nhanh và giữ đúng kiểu dữ liệu; CSV là bản để đọc và nộp. Khi đọc lại CSV, luôn ép kiểu thời gian bằng `parse_dates` và kiểm tra số dòng trùng khớp với manifest.

In [9]:
# Export the standardised raw snapshot to CSV and read it back to confirm the round trip (dtypes and row count)
RAW_CSV = "data/raw/primary_snapshot.csv"
raw.to_csv(RAW_CSV, index=False)
check = pd.read_csv(RAW_CSV, parse_dates=[TIME_COL])
assert len(check) == len(raw), "row count changed in the CSV round trip"
print("wrote", RAW_CSV, "| rows", len(check), "| columns", list(check.columns)[:8])

wrote data/raw/primary_snapshot.csv | rows 99995 | columns ['station', 'date', 'wtemp', 'dox', 'turb', 'flow']


### Bước 1d: thu thập thêm dữ liệu (mở rộng phạm vi thời gian hoặc đơn vị)

**Làm gì**: nếu nguồn cho phép, tải thêm giai đoạn hoặc thêm đơn vị (trạm, vùng) để bảng đủ khoảng 100 nghìn dòng và có đủ ít nhất hai mùa hoặc hai chế độ. Hàm dưới ghép nhiều file tải theo năm hoặc theo đơn vị và loại trùng. **Vì sao**: mô hình toàn cục và kiểm thử đơn vị chưa thấy cần nhiều đơn vị; đổi chế độ chỉ đo được khi có dữ liệu trước và sau.

In [10]:
# Step 1d: combine several downloaded chunks (data/raw/primary_*.csv) into one table, deduplicated on unit and time
import glob
chunks = sorted(glob.glob(PRIMARY_SOURCE["path"].replace(".csv", "_*.csv")))
if chunks:
    extra = pd.concat([load_table(f) for f in chunks]); extra[TIME_COL] = pd.to_datetime(extra[TIME_COL], errors="coerce")
    before = len(raw); raw = pd.concat([raw, extra]).drop_duplicates([UNIT_COL, TIME_COL] if UNIT_COL in raw.columns else [TIME_COL])
    print(f"added {len(raw) - before} rows from {len(chunks)} chunks")
else:
    print("no extra chunks found (pattern primary_*.csv); skip if your source is a single file")
if UNIT_COL not in raw.columns: raw[UNIT_COL] = "all"; print("single-series data: UNIT_COL set to 'all'")

added 0 rows from 1 chunks


## Bước 2: so sánh hai nguồn và làm sạch có nhật ký

**So sánh nguồn (RQ1a)**: trước khi ghép, đo (1) khoảng thời gian chồng nhau, (2) tỷ lệ bước thời gian có ở nguồn này mà không có ở nguồn kia, (3) nếu hai nguồn có cùng một biến thì tương quan và độ lệch trung bình, (4) độ trễ cập nhật (thời điểm cuối của mỗi nguồn). Bảng này vào phần Data của bài.

**Làm sạch có nhật ký**: mỗi thao tác ghi số dòng trước, sau và lý do; bảng nhật ký là bằng chứng để người đọc tin dữ liệu.

In [11]:
# Step 2a: source comparison table
def compare_sources(a, b, time_col, common_var=None):
    ta, tb = a[time_col].dropna(), b[time_col].dropna()
    ov_start, ov_end = max(ta.min(), tb.min()), min(ta.max(), tb.max())
    ga = set(ta.dt.floor(FREQ).unique()); gb = set(tb.dt.floor(FREQ).unique())
    rows = [["overlap start", ov_start], ["overlap end", ov_end], ["steps only in primary", len(ga - gb)], ["steps only in secondary", len(gb - ga)],
            ["latest timestamp primary", ta.max()], ["latest timestamp secondary", tb.max()]]
    if common_var and common_var in a.columns and common_var in b.columns:
        m = a.groupby(time_col)[common_var].mean().to_frame("a").join(b.groupby(time_col)[common_var].mean().to_frame("b"), how="inner")
        rows += [["correlation of common variable", round(m.a.corr(m.b), 3)], ["mean difference (primary minus secondary)", round((m.a - m.b).mean(), 3)]]
    return pd.DataFrame(rows, columns=["metric", "value"])
cmp_tbl = compare_sources(raw, sec, TIME_COL, common_var=None)     # set common_var="temp" if both sources carry the same variable
cmp_tbl.to_csv("report/table_source_comparison.csv", index=False); display(cmp_tbl)

,metric,value
0,overlap start,2018-01-01 00:00:00
1,overlap end,2026-09-14 00:00:00
2,steps only in primary,0
3,steps only in secondary,0
4,latest timestamp primary,2026-09-14 00:00:00
5,latest timestamp secondary,2026-09-14 00:00:00


In [12]:
# Step 2b: cleaning with a log; each step records rows before and after and the reason
clean_log = []
def step(df, name, fn, reason):
    n0 = len(df); out = fn(df); clean_log.append([name, n0, len(out), n0 - len(out), reason]); return out
df = raw.copy(); df[UNIT_COL] = df[UNIT_COL].astype(str)
df = step(df, "parse time", lambda x: x.dropna(subset=[TIME_COL]), "rows with unparseable timestamps")
df = step(df, "drop duplicates", lambda x: x.drop_duplicates([UNIT_COL, TIME_COL]), "same unit and timestamp")
for c in [TARGET_COL] + PRIM_COVS:
    if c in df.columns: df[c] = pd.to_numeric(df[c], errors="coerce")
df = step(df, "drop missing target", lambda x: x.dropna(subset=[TARGET_COL]), "target missing (never filled)")
lo, hi = PHYSICAL_RANGES[TARGET_COL]
df = step(df, "physical range (target)", lambda x: x[(x[TARGET_COL] >= lo) & (x[TARGET_COL] <= hi)], f"outside the physical range [{lo}, {hi}]")
n_bad = 0                                                   # other parameters: out-of-range values become NaN, the row is kept
for c in PRIM_COVS:
    if c in df.columns and c in PHYSICAL_RANGES:
        l, h = PHYSICAL_RANGES[c]; bad = (df[c] < l) | (df[c] > h); n_bad += int(bad.sum()); df.loc[bad, c] = np.nan
clean_log.append(["covariates outside physical range set to NaN", len(df), len(df), 0, f"{n_bad} values, rows kept"])
pd.DataFrame(clean_log, columns=["step", "rows_before", "rows_after", "rows_removed", "reason"]).to_csv("report/table_cleaning_log.csv", index=False)
display(pd.DataFrame(clean_log, columns=["step", "rows_before", "rows_after", "rows_removed", "reason"]))

,step,rows_before,rows_after,rows_removed,reason
0,parse time,99995,99995,0,rows with unparseable timestamps
1,drop duplicates,99995,99995,0,same unit and timestamp
2,drop missing target,99995,82927,17068,target missing (never filled)
3,physical range (target),82927,82927,0,"outside the physical range [-2, 40]"
4,covariates outside physical range set to NaN,82927,82927,0,"0 values, rows kept"


In [13]:
# Step 2c: regular daily grid per station, then short-gap interpolation of the TARGET only (limit = 3 steps, never across long gaps)
# The full grid (df_grid) is kept: LAG/LEAD/rolling windows in Step 3 must run on calendar days, not on whatever rows survived cleaning.
num_cols = [c for c in df.columns if c not in (UNIT_COL, TIME_COL) and pd.api.types.is_numeric_dtype(df[c])]
n_before = len(df)
df = df.set_index(TIME_COL).groupby(UNIT_COL)[num_cols].resample(FREQ).mean().reset_index()
gap = df.groupby(UNIT_COL)[TARGET_COL].apply(lambda s: s.isna().mean()).rename("missing_share")
df[TARGET_COL] = df.groupby(UNIT_COL)[TARGET_COL].transform(lambda s: s.interpolate(limit=3))   # covariates are NOT interpolated (no look-ahead); Notebook_D adds missing flags
clean_log.append(["regular grid + interpolate target (limit=3)", n_before, len(df), n_before - len(df), "rows_removed < 0 means empty grid rows were added; target gaps longer than 3 steps stay NaN in the grid"])
df_grid = df.copy()
print("missing share of the target per station before interpolation:"); display(gap.describe().round(3))

missing share of the target per station before interpolation:


count    37.000
mean      0.044
std       0.091
min       0.000
25%       0.005
50%       0.010
75%       0.023
max       0.395
Name: missing_share, dtype: float64

In [14]:
# Step 2d: keep stations with enough history (at least MIN_UNIT_STEPS days with a target) so the global model is not dominated by fragments
cnt = df_grid.dropna(subset=[TARGET_COL]).groupby(UNIT_COL).size(); keep = cnt[cnt >= MIN_UNIT_STEPS].index
n_before = len(df_grid); df_grid = df_grid[df_grid[UNIT_COL].isin(keep)].copy()
clean_log.append(["drop short stations", n_before, len(df_grid), n_before - len(df_grid), f"{len(cnt) - len(keep)} stations with fewer than {MIN_UNIT_STEPS} target days"])
print("stations kept:", len(keep), "of", len(cnt))

stations kept: 37 of 37


In [15]:
# Step 2e: merge the secondary source (nearest timestamp within one step, by station) onto the full grid
sec_num = sec[[TIME_COL, UNIT_COL] + [c for c in EXOG_COLS if c in sec.columns]].copy()
sec_num[UNIT_COL] = sec_num[UNIT_COL].astype(str)
for c in EXOG_COLS:
    if c in sec_num.columns: sec_num[c] = pd.to_numeric(sec_num[c], errors="coerce")
sec_num = sec_num.sort_values(TIME_COL); df_grid = df_grid.sort_values(TIME_COL)
tol = pd.Timedelta(pd.tseries.frequencies.to_offset(FREQ))
m = pd.merge_asof(df_grid, sec_num, on=TIME_COL, by=UNIT_COL, direction="nearest", tolerance=tol)
df_grid = m.sort_values([UNIT_COL, TIME_COL]).reset_index(drop=True)
n_before = len(df_grid); df = df_grid.dropna(subset=[TARGET_COL]).reset_index(drop=True)
clean_log.append(["drop target NaN after interpolation (grid kept for lags)", n_before, len(df), n_before - len(df), "gaps longer than 3 steps"])
COVS = [c for c in EXOG_COLS + PRIM_COVS if c in df.columns]       # every covariate: secondary source first, then other NWIS parameters
print("share of rows with each covariate:", {c: round(df[c].notna().mean(), 3) for c in COVS})

share of rows with each covariate: {'TMAX': np.float64(0.959), 'PRCP': np.float64(0.944), 'dox': np.float64(0.871), 'turb': np.float64(0.922), 'flow': np.float64(0.99)}


In [16]:
# Step 2f: save the clean table, the full grid and the cleaning log; print a one-paragraph data statement for the paper
df.to_parquet("data/processed/clean.parquet", index=False)
df_grid.to_parquet("data/processed/clean_grid.parquet", index=False)                 # full daily grid: Notebook_D builds foundation-model contexts from it
df.to_csv("data/processed/clean.csv", index=False)                                   # separate clean CSV for submission and for opening in Excel
chk = pd.read_csv("data/processed/clean.csv", parse_dates=[TIME_COL], dtype={UNIT_COL: str}); assert len(chk) == len(df) and chk[TARGET_COL].notna().all(), "clean.csv round trip failed"
print("wrote data/processed/clean.csv with", len(chk), "rows")
pd.DataFrame(clean_log, columns=["step", "rows_before", "rows_after", "rows_removed", "reason"]).to_csv("report/table_cleaning_log.csv", index=False)
print(f"Data statement: {PRIMARY_SOURCE['name']} ({PRIMARY_SOURCE['license']}) merged with {SECOND_SOURCE['name']}; {len(df):,} station-days with a target, {df[UNIT_COL].nunique()} stations, "
      f"{df[TIME_COL].min().date()} to {df[TIME_COL].max().date()} at frequency {FREQ}; {sum(max(r[3], 0) for r in clean_log):,} rows removed by cleaning.")

wrote data/processed/clean.csv with 83722 rows
Data statement: USGS NWIS daily values (dataretrieval) (US public domain) merged with NOAA GHCN-Daily, nearest station (PRCP, TMAX); 83,722 station-days with a target, 37 stations, 2018-01-01 to 2026-09-14 at frequency D; 19,790 rows removed by cleaning.


## EDA (RQ1): mô tả, mùa vụ, đơn vị, nguồn phụ, tự tương quan

**Cách đọc**: mỗi hình trả lời một câu hỏi nhỏ và ghi một nhận xét có thể kiểm chứng trong caption. Không vẽ hình chỉ để đẹp. Thứ tự: (1) tổng quan phân bố, (2) chuỗi theo thời gian của vài đơn vị, (3) mùa vụ theo lịch, (4) khác biệt giữa đơn vị, (5) quan hệ với nguồn phụ và độ trễ, (6) tự tương quan quyết định đặc trưng trễ, (7) thiếu dữ liệu.

In [17]:
# EDA 1: automatic profile (open report/profile.html in a browser) and summary statistics
from ydata_profiling import ProfileReport
sample = df.sample(min(20000, len(df)), random_state=42)
ProfileReport(sample, title=f"Profile {GROUP}", minimal=True).to_file("report/profile.html")
desc = df[[TARGET_COL] + COVS].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T.round(3)
desc.to_csv("report/table_describe.csv"); display(desc)

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 8/8 [00:00<00:00, 528.82it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

,count,mean,std,min,1%,5%,50%,95%,99%,max
wtemp,83722.0,14.060,8.168,-0.50,0.00,1.10,14.3,26.30,28.8,33.1
TMAX,80263.0,18.231,10.326,-16.10,-5.00,0.60,19.4,32.50,35.6,40.0
PRCP,79048.0,3.133,8.346,0.00,0.00,0.00,0.0,18.50,40.4,168.7
dox,72923.0,10.273,2.145,3.50,6.50,7.40,9.9,14.10,14.9,17.3
turb,77155.0,12.185,36.607,0.00,0.40,0.80,4.0,46.33,150.0,1638.0
flow,82877.0,1775.789,4918.125,0.12,1.62,4.21,164.0,9130.00,21624.0,148000.0


In [18]:
# EDA 2: distribution of the target (histogram with many bins) and log-scale check for skewed data
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].hist(df[TARGET_COL], bins=60, color=TEAL); axes[0].set_xlabel(TARGET_COL); axes[0].set_ylabel("Count"); style(axes[0])
pos = df[TARGET_COL][df[TARGET_COL] > 0]
axes[1].hist(np.log1p(pos), bins=60, color=ACC); axes[1].set_xlabel("log1p(" + TARGET_COL + ")"); style(axes[1])
savefig(fig, "fig_eda_distribution"); print("skewness:", round(df[TARGET_COL].skew(), 2), "-> consider modelling log1p if skewness > 2")

saved report/fig_eda_distribution.png
skewness: -0.04 -> consider modelling log1p if skewness > 2


In [19]:
# EDA 3: time series of the first four units over the whole period (look for trends, regime changes, gaps)
units = df[UNIT_COL].unique()[:4]
fig, ax = plt.subplots(figsize=(11, 3.4))
for u, col in zip(units, [TEAL, ACC, GREY, "#264653"]):
    g = df[df[UNIT_COL] == u]; ax.plot(g[TIME_COL], g[TARGET_COL], lw=0.7, color=col, label=str(u))
ax.axvline(pd.Timestamp(TEST_START), color="k", ls="--", lw=1); ax.set_ylabel(TARGET_COL); ax.legend(frameon=False, ncol=4); style(ax)
savefig(fig, "fig_eda_timeseries")

saved report/fig_eda_timeseries.png


In [20]:
# EDA 4: calendar seasonality: mean target by hour of day, day of week and month (only the keys that vary at the daily frequency)
t = df[TIME_COL]; cal = pd.DataFrame({"hour": t.dt.hour, "dow": t.dt.dayofweek, "month": t.dt.month, TARGET_COL: df[TARGET_COL]})
keys = [k for k in ["hour", "dow", "month"] if cal[k].nunique() > 1]
fig, axes = plt.subplots(1, len(keys), figsize=(4.5 * len(keys), 3.2)); axes = np.atleast_1d(axes)
for ax, k in zip(axes, keys):
    s = cal.groupby(k)[TARGET_COL].mean(); ax.plot(s.index, s.values, marker="o", color=TEAL); ax.set_xlabel(k); style(ax)
axes[0].set_ylabel("mean " + TARGET_COL); savefig(fig, "fig_eda_seasonality")
season_strength = {k: round(cal.groupby(k)[TARGET_COL].mean().std() / cal[TARGET_COL].std(), 3) for k in keys}
print("seasonal strength (std of group means / total std):", season_strength)

saved report/fig_eda_seasonality.png
seasonal strength (std of group means / total std): {'dow': np.float64(0.009), 'month': np.float64(0.974)}


In [21]:
# EDA 5: heterogeneity across units: mean and variability per unit, sorted (global models must handle this spread)
per_unit = df.groupby(UNIT_COL)[TARGET_COL].agg(["mean", "std", "min", "max", "count"]).sort_values("mean")
per_unit.to_csv("report/table_per_unit.csv")
fig, ax = plt.subplots(figsize=(10, 3.2)); ax.bar(range(len(per_unit)), per_unit["mean"], color=TEAL, yerr=None)
ax.set_xlabel("units sorted by mean"); ax.set_ylabel("mean " + TARGET_COL); style(ax); savefig(fig, "fig_eda_units")
print("ratio of largest to smallest unit mean:", round(per_unit["mean"].max() / max(per_unit["mean"].min(), 1e-9), 2))

saved report/fig_eda_units.png
ratio of largest to smallest unit mean: 2.0


In [22]:
# EDA 6: relation with covariates: binned means and Spearman correlation (robust to outliers)
from scipy.stats import spearmanr
rows = []
for c in [c for c in EXOG_COLS if c in df.columns]:
    ok = df[[c, TARGET_COL]].dropna(); rho, p = spearmanr(ok[c], ok[TARGET_COL]); rows.append([c, round(rho, 3), p, len(ok)])
    fig, ax = plt.subplots(figsize=(5.5, 3.2)); b = ok.groupby(pd.qcut(ok[c], 10, duplicates="drop"))[TARGET_COL].mean()
    ax.plot(range(len(b)), b.values, marker="o", color=TEAL); ax.set_xlabel(c + " (deciles)"); ax.set_ylabel("mean " + TARGET_COL); style(ax); savefig(fig, "fig_eda_covariate_" + c)
corr_tbl = pd.DataFrame(rows, columns=["covariate", "spearman_rho", "p_value", "n"]); corr_tbl.to_csv("report/table_covariate_corr.csv", index=False); display(corr_tbl)

saved report/fig_eda_covariate_TMAX.png
saved report/fig_eda_covariate_PRCP.png


,covariate,spearman_rho,p_value,n
0,TMAX,0.920,0.000000,80263
1,PRCP,0.006,0.111226,79048


In [23]:
# EDA 7: cross-correlation with lags: does the covariate lead the target? (positive lag = covariate earlier). Uses the full grid so a lag is a calendar lag.
def cross_corr(g, x, y, lags):
    return [g[y].corr(g[x].shift(k)) for k in lags]
lags = list(range(0, 4 * SEASON + 1, max(1, SEASON // 4)))
fig, ax = plt.subplots(figsize=(7, 3.2))
for c in COVS[:5]:
    cc = np.nanmean([cross_corr(g, c, TARGET_COL, lags) for _, g in df_grid.groupby(UNIT_COL) if len(g) > 5 * SEASON], axis=0)
    ax.plot(lags, cc, marker="o", label=c); print(c, "best lag:", lags[int(np.nanargmax(np.abs(cc)))], "corr:", round(float(np.nanmax(np.abs(cc))), 3))
ax.axhline(0, color=GREY); ax.set_xlabel("lag (days, covariate leads target)"); ax.set_ylabel("correlation"); ax.legend(frameon=False); style(ax); savefig(fig, "fig_eda_crosscorr")

TMAX best lag: 0 corr: 0.928
PRCP best lag: 21 corr: 0.049
dox best lag: 0 corr: 0.942
turb best lag: 28 corr: 0.043
flow best lag: 1 corr: 0.141
saved report/fig_eda_crosscorr.png


In [24]:
# EDA 8: autocorrelation of the target averaged over stations on the full grid (pandas autocorr handles NaN pairwise): which lags to use as features
nl = 2 * SEASON + 1
acfs = [[g[TARGET_COL].autocorr(k) for k in range(nl + 1)] for _, g in df_grid.groupby(UNIT_COL) if g[TARGET_COL].notna().sum() > 4 * SEASON]
acf_mean = np.nanmean(acfs, axis=0)
fig, ax = plt.subplots(figsize=(8, 3.2)); ax.stem(range(nl + 1), acf_mean, basefmt=" "); ax.set_xlabel("lag (days)"); ax.set_ylabel("ACF"); style(ax); savefig(fig, "fig_eda_acf")
print("ACF at lag 1:", round(acf_mean[1], 3), "| at SEASON:", round(acf_mean[SEASON], 3), "-> persistence is very strong when ACF at lag 1 is close to 1; the annual cycle is handled by doy features")

saved report/fig_eda_acf.png
ACF at lag 1: 0.989 | at SEASON: 0.932 -> persistence is very strong when ACF at lag 1 is close to 1; the annual cycle is handled by doy features


In [25]:
# EDA 9: missingness pattern over time per station (heatmap of monthly missing share, full grid) and regime check (deseasonalised monthly anomaly)
miss = df_grid.set_index(TIME_COL).groupby(UNIT_COL)[TARGET_COL].resample("MS").apply(lambda s: s.isna().mean()).unstack(0)
fig, ax = plt.subplots(figsize=(10, 4.2)); sns.heatmap(miss.T, cmap="Blues", cbar_kws={"label": "missing share"}, ax=ax, xticklabels=12); ax.set_xlabel("month"); ax.set_ylabel(UNIT_COL); savefig(fig, "fig_eda_missing")
# water temperature has a huge seasonal cycle, so a raw monthly mean would always look like a regime shift: subtract the month-of-year mean first
anom = df[TARGET_COL] - df.groupby(df[TIME_COL].dt.month)[TARGET_COL].transform("mean")
roll = anom.groupby(df[TIME_COL].dt.to_period("M").dt.to_timestamp()).mean()
fig, ax = plt.subplots(figsize=(10, 3.0)); ax.plot(roll.index, roll.values, color=TEAL, marker="o", ms=3); ax.axvline(pd.Timestamp(TEST_START), color="k", ls="--"); ax.axhline(0, color=GREY); ax.set_ylabel("monthly anomaly of " + TARGET_COL + " (C)"); style(ax); savefig(fig, "fig_eda_regime")
print("largest monthly anomaly (C):", round(float(roll.abs().max()), 2), "at", roll.abs().idxmax().date(), "-> above about 1.5 C suggests a regime shift or a change in station mix worth reporting in RQ3")

saved report/fig_eda_missing.png
saved report/fig_eda_regime.png
largest monthly anomaly (C): 2.75 at 2018-04-01 -> above about 1.5 C suggests a regime shift or a change in station mix worth reporting in RQ3


### Ghi nhận xét EDA (bắt buộc, dùng cho phần Results RQ1)

Điền 5 nhận xét, mỗi nhận xét chỉ đúng một con số ở bảng hoặc hình phía trên, ví dụ: "Sức mạnh mùa theo giờ là 0,42 trong khi theo tháng chỉ 0,08, nên đặc trưng giờ quan trọng hơn tháng (Hình EDA 4)". Không viết nhận xét không có số.

In [26]:
# EDA 10: write your five evidence-backed observations here (they go straight into the paper)
eda_notes = [
    "1. ...",
    "2. ...",
    "3. ...",
    "4. ...",
    "5. ...",
]
open("report/eda_notes.md", "w").write("\n".join(eda_notes)); print("saved report/eda_notes.md")

saved report/eda_notes.md


## Bước 3: SQL với DuckDB, trả lời RQ1 và tạo bảng đặc trưng

**Vì sao SQL**: người đọc bài kiểm tra được từng con số bằng một truy vấn; đặc trưng trễ và cửa sổ trượt bằng `WINDOW` không rò rỉ tương lai nếu chỉ dùng `LAG` và `PRECEDING`.

**Quy tắc chống rò rỉ**: đặc trưng chỉ dùng thông tin **tại hoặc trước** thời điểm t; mục tiêu là `LEAD(target, h)`; không dùng thống kê tính trên toàn chuỗi (kể cả trung bình đơn vị) trừ khi tính chỉ trên phần huấn luyện.

In [27]:
# Step 3a: register the tables and write the RQ1 queries to sql/queries.sql
# clean = rows with a target; clean_grid = full daily grid (use it for LAG/LEAD so one step is one calendar day)
con.register("clean", df); con.register("clean_grid", df_grid)
EXOG_SQL = ", ".join(COVS)
THR_SQL = f"SELECT {EVENT_ABS} AS q" if EVENT_ABS is not None else f"SELECT quantile_cont({TARGET_COL}, {EVENT_QUANTILE}) AS q FROM clean"
EVENT_TXT = f"above {EVENT_ABS} C" if EVENT_ABS is not None else f"above the {EVENT_QUANTILE:.0%} quantile"
SQL_RQ1 = f"""
-- Q1: mean and spread of the target per station and calendar month
SELECT {UNIT_COL}, month({TIME_COL}) AS mon, AVG({TARGET_COL}) AS mean_target, STDDEV({TARGET_COL}) AS sd_target, COUNT(*) AS n FROM clean GROUP BY 1, 2 ORDER BY 1, 2;
-- Q2: share of event days (target {EVENT_TXT}) per station
WITH thr AS ({THR_SQL})
SELECT {UNIT_COL}, AVG(({TARGET_COL} > q)::INT) AS event_share FROM clean, thr GROUP BY 1 ORDER BY 2 DESC;
-- Q3: target by covariate decile (relationship with the secondary source)
SELECT decile, AVG({TARGET_COL}) AS mean_target, COUNT(*) AS n FROM (SELECT NTILE(10) OVER (ORDER BY {EXOG_COLS[0] if EXOG_COLS else TARGET_COL}) AS decile, {TARGET_COL} FROM clean WHERE {EXOG_COLS[0] if EXOG_COLS else TARGET_COL} IS NOT NULL) GROUP BY 1 ORDER BY 1;
-- Q4: seasonal-naive skill check: correlation between the target and its value SEASON days earlier (full grid)
SELECT corr({TARGET_COL}, lagged) AS r_season FROM (SELECT {TARGET_COL}, LAG({TARGET_COL}, {SEASON}) OVER (PARTITION BY {UNIT_COL} ORDER BY {TIME_COL}) AS lagged FROM clean_grid);
"""
if "PRCP" in df.columns and "turb" in df.columns:
    SQL_RQ1 += f"""-- Q5: storm response: turbidity on the day after rain above 10 mm versus the day after dry days
SELECT (PRCP > 10) AS rain, AVG(turb_next) AS turb_next_day, COUNT(*) AS n FROM (SELECT PRCP, LEAD(turb, 1) OVER (PARTITION BY {UNIT_COL} ORDER BY {TIME_COL}) AS turb_next FROM clean_grid) WHERE PRCP IS NOT NULL GROUP BY 1 ORDER BY 1;
"""
if "dox" in df.columns:
    SQL_RQ1 += f"""-- Q6: low-oxygen days (dissolved oxygen below 5 mg/L) and turbidity peaks per station and year
SELECT {UNIT_COL}, year({TIME_COL}) AS yr, SUM((dox < 5)::INT) AS low_do_days, MAX(turb) AS turb_max, AVG({TARGET_COL}) AS wtemp_mean FROM clean_grid WHERE dox IS NOT NULL GROUP BY 1, 2 HAVING SUM((dox < 5)::INT) > 0 ORDER BY 1, 2;
"""
open("sql/queries.sql", "w").write(SQL_RQ1); print(SQL_RQ1)


-- Q1: mean and spread of the target per station and calendar month
SELECT station, month(date) AS mon, AVG(wtemp) AS mean_target, STDDEV(wtemp) AS sd_target, COUNT(*) AS n FROM clean GROUP BY 1, 2 ORDER BY 1, 2;
-- Q2: share of event days (target above 20.0 C) per station
WITH thr AS (SELECT 20.0 AS q)
SELECT station, AVG((wtemp > q)::INT) AS event_share FROM clean, thr GROUP BY 1 ORDER BY 2 DESC;
-- Q3: target by covariate decile (relationship with the secondary source)
SELECT decile, AVG(wtemp) AS mean_target, COUNT(*) AS n FROM (SELECT NTILE(10) OVER (ORDER BY TMAX) AS decile, wtemp FROM clean WHERE TMAX IS NOT NULL) GROUP BY 1 ORDER BY 1;
-- Q4: seasonal-naive skill check: correlation between the target and its value SEASON days earlier (full grid)
SELECT corr(wtemp, lagged) AS r_season FROM (SELECT wtemp, LAG(wtemp, 7) OVER (PARTITION BY station ORDER BY date) AS lagged FROM clean_grid);
-- Q5: storm response: turbidity on the day after rain above 10 mm versus the day after dry 

In [28]:
# Step 3b: run every statement, save each result as report/table_q{k}.csv (comment lines removed before splitting on ';')
def run_sql_file(path):
    txt = open(path).read(); body = "\n".join(l for l in txt.splitlines() if not l.strip().startswith("--"))
    k = 0
    for stmt in body.split(";"):
        stmt = stmt.strip()
        if not stmt: continue
        k += 1; out = con.execute(stmt).df(); out.to_csv(f"report/table_q{k}.csv", index=False); print(f"Q{k}: {out.shape}"); display(out.head(8))
run_sql_file("sql/queries.sql")

Q1: (436, 5)


,station,mon,mean_target,sd_target,n
0,01438500,1,1.327189,1.490673,217
1,01438500,2,1.314646,1.431707,198
2,01438500,3,4.882028,2.277080,217
3,01438500,4,10.212500,2.816798,206
4,01438500,5,15.915207,3.363736,217
5,01438500,6,21.459048,2.484986,210
6,01438500,7,24.441935,1.833692,217
7,01438500,8,23.252558,1.956907,215


Q2: (37, 2)


,station,event_share
0,01473500,0.451137
1,01643580,0.409207
2,01638500,0.400307
3,01646500,0.394255
4,03185400,0.394205
5,04213152,0.393425
6,01473900,0.390152
7,01491000,0.386967


Q3: (10, 3)


...

,r_season
0,0.944626


Q5: (2, 3)


,rain,turb_next_day,n
0,False,10.862082,73341
1,True,26.018695,8210


Q6: (5, 5)


,station,yr,low_do_days,turb_max,wtemp_mean
0,01473900,2024,3.0,58.0,17.188235
1,01473900,2026,1.0,51.1,19.447429
2,01491000,2025,1.0,34.0,14.465014
3,01491000,2026,6.0,25.9,16.665234
4,01649500,2019,13.0,139.0,15.342183


In [29]:
# Step 3c: statistical tests behind RQ1 (report p-values, not only means): Kruskal-Wallis across stations and across months, Spearman with covariates
from scipy.stats import kruskal
groups = [g[TARGET_COL].values for _, g in df.groupby(UNIT_COL) if len(g) > 30]
H, p = kruskal(*groups) if len(groups) > 1 else (np.nan, np.nan)
tests = [["Kruskal-Wallis target across stations", round(H, 2) if H == H else "n/a", p]]
H2, p2 = kruskal(*[g[TARGET_COL].values for _, g in df.groupby(df[TIME_COL].dt.month)])
tests.append(["Kruskal-Wallis target across months", round(H2, 2), p2])
for c in COVS:
    ok = df[[c, TARGET_COL]].dropna(); rho, pv = spearmanr(ok[c], ok[TARGET_COL]); tests.append([f"Spearman target vs {c}", round(rho, 3), pv])
tests = pd.DataFrame(tests, columns=["test", "statistic", "p_value"]); tests.to_csv("report/table_rq1_tests.csv", index=False); display(tests)

,test,statistic,p_value
0,Kruskal-Wallis target across stations,4155.140,0.000000e+00
1,Kruskal-Wallis target across months,72118.560,0.000000e+00
2,Spearman target vs TMAX,0.920,0.000000e+00
3,Spearman target vs PRCP,0.006,1.112262e-01
4,Spearman target vs dox,-0.936,0.000000e+00
5,Spearman target vs turb,0.019,9.935150e-08
6,Spearman target vs flow,-0.044,9.383106e-37


In [30]:
# Step 3d: build the feature table with SQL on the FULL daily grid (lags, rolling means, calendar, covariates, targets for every horizon); no future information
# Seasonal naive for horizon h needs the value SEASON - h days before the forecast origin, so those lags are added explicitly (y_lag{SEASON-h}).
lag_list = sorted(set([1, 2, 3, SEASON, 2 * SEASON, 4 * SEASON] + [SEASON - h for h in HORIZONS if SEASON - h > 0]))
lag_sql = ", ".join([f"LAG({TARGET_COL}, {k}) OVER w AS y_lag{k}" for k in lag_list])
roll_sql = (f"AVG({TARGET_COL}) OVER (w ROWS BETWEEN {SEASON - 1} PRECEDING AND CURRENT ROW) AS y_ma_season, "
            f"STDDEV({TARGET_COL}) OVER (w ROWS BETWEEN {SEASON - 1} PRECEDING AND CURRENT ROW) AS y_sd_season, "
            f"{TARGET_COL} - LAG({TARGET_COL}, 1) OVER w AS y_diff1")
cov_roll = ", ".join(f"AVG({c}) OVER (w ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS {c}_roll3" for c in COVS)       # 3-day mean of every covariate (past only)
exog_lag1 = ", ".join(f"LAG({c}, 1) OVER w AS {c}_lag1" for c in EXOG_COLS if c in COVS)                             # yesterday's weather
targets = ", ".join([f"LEAD({TARGET_COL}, {h}) OVER w AS y_h{h}" for h in HORIZONS])
cal_sql = (f"dayofweek({TIME_COL}) AS dow, month({TIME_COL}) AS mon, dayofyear({TIME_COL}) AS doy, "
           f"sin(2 * pi() * dayofyear({TIME_COL}) / 365.25) AS doy_sin, cos(2 * pi() * dayofyear({TIME_COL}) / 365.25) AS doy_cos"
           + (f", hour({TIME_COL}) AS hr" if FREQ not in ("D", "W", "M", "MS") else ""))
select_parts = [UNIT_COL, TIME_COL, TARGET_COL, EXOG_SQL, cal_sql, lag_sql, roll_sql, cov_roll, exog_lag1, targets]
FEAT_SQL = f"""
CREATE OR REPLACE TABLE feat AS
SELECT * FROM (
  SELECT {", ".join(p for p in select_parts if p)}
  FROM clean_grid WINDOW w AS (PARTITION BY {UNIT_COL} ORDER BY {TIME_COL})
) WHERE {TARGET_COL} IS NOT NULL;
"""
con.execute(FEAT_SQL); open("sql/features.sql", "w").write(FEAT_SQL)
feat = con.execute("SELECT * FROM feat").df().sort_values([UNIT_COL, TIME_COL]).reset_index(drop=True)
print(feat.shape); display(feat.head(3))

(83722, 33)


,station,date,wtemp,TMAX,PRCP,dox,turb,flow,dow,mon,doy,doy_sin,doy_cos,y_lag1,y_lag2,y_lag3,y_lag4,y_lag6,y_lag7,y_lag14,y_lag28,y_ma_season,y_sd_season,y_diff1,TMAX_roll3,PRCP_roll3,dox_roll3,turb_roll3,flow_roll3,TMAX_lag1,PRCP_lag1,y_h1,y_h3
0,01438500,2019-10-03,19.0,29.4,0.0,8.6,0.9,1660.0,4,10,276,-0.999371,0.035473,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.000000,NaN,NaN,29.400000,0.000000,8.600000,0.9,1660.000000,NaN,NaN,16.9,14.4
1,01438500,2019-10-04,16.9,11.7,9.4,9.6,0.9,1470.0,5,10,277,-0.998613,0.052658,19.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.950000,1.484924,-2.1,20.550000,4.700000,9.100000,0.9,1565.000000,29.4,0.0,15.3,14.7
2,01438500,2019-10-05,15.3,13.9,0.0,10.4,0.9,1680.0,6,10,278,-0.997559,0.069828,16.9,19.0,NaN,NaN,NaN,NaN,NaN,NaN,17.066667,1.855622,-1.6,18.333333,3.133333,9.533333,0.9,1603.333333,11.7,9.4,14.4,15.1


In [31]:
# Step 3e: leakage check against the raw daily grid: every y_h{h} must equal the target h calendar days later, every y_lag{k} the target k days earlier
grid = df_grid.set_index([UNIT_COL, TIME_COL])[TARGET_COL]
step_td = pd.Timedelta(pd.tseries.frequencies.to_offset(FREQ))
for h in HORIZONS:
    chk = feat.dropna(subset=[f"y_h{h}"]); chk = chk.sample(min(5000, len(chk)), random_state=0)
    idx = pd.MultiIndex.from_arrays([chk[UNIT_COL], chk[TIME_COL] + step_td * h])
    ok = np.allclose(grid.reindex(idx).values, chk[f"y_h{h}"].values, equal_nan=True); print(f"horizon {h}: target equals the raw value {h} step(s) later:", ok); assert ok, "target alignment broken"
for k in sorted(set([1, SEASON])):
    chk = feat.dropna(subset=[f"y_lag{k}"]); chk = chk.sample(min(5000, len(chk)), random_state=0)
    idx = pd.MultiIndex.from_arrays([chk[UNIT_COL], chk[TIME_COL] - step_td * k])
    ok = np.allclose(grid.reindex(idx).values, chk[f"y_lag{k}"].values, equal_nan=True); print(f"lag {k}: feature equals the raw value {k} step(s) earlier:", ok); assert ok, "lag alignment broken"

horizon 1: target equals the raw value 1 step(s) later: True
horizon 3: target equals the raw value 3 step(s) later: True
lag 1: feature equals the raw value 1 step(s) earlier: True
lag 7: feature equals the raw value 7 step(s) earlier: True


In [32]:
# Step 3f: RQ1 figures 2 and 3 for the paper: event share per unit and target by covariate decile (from the SQL outputs)
q2 = pd.read_csv("report/table_q2.csv"); q3 = pd.read_csv("report/table_q3.csv")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
b = axes[0].bar(range(len(q2)), q2.event_share, color=TEAL); axes[0].set_xlabel("units"); axes[0].set_ylabel("event share"); style(axes[0])
axes[1].plot(q3.decile, q3.mean_target, marker="o", color=ACC); axes[1].set_xlabel("covariate decile"); axes[1].set_ylabel("mean " + TARGET_COL); style(axes[1])
savefig(fig, "fig_rq1_sql")

saved report/fig_rq1_sql.png


## Kiểm tra, bàn giao và ghi Audit Log

**Test case** phải chạy qua trước khi bàn giao. **Manifest** ghi số dòng, cột, khoảng thời gian và mã MD5 của parquet để Sinh viên B xác nhận nhận đúng file. Sau đó điền **AI Audit Log** cho các prompt đã dùng ở tuần 1 và 2 (ít nhất 3 mục mỗi tuần, có ít nhất một mục phát hiện AI sai).

In [33]:
# Test cases for Notebook_C (all asserts must pass)
assert feat[[UNIT_COL, TIME_COL]].duplicated().sum() == 0, "duplicate unit-time rows"
assert feat[TARGET_COL].notna().all(), "target missing after cleaning"
assert set(HORIZONS) <= set(int(c[3:]) for c in feat.columns if c.startswith("y_h")), "horizon targets missing"
assert (feat.groupby(UNIT_COL)[TIME_COL].apply(lambda s: s.is_monotonic_increasing)).all(), "time not sorted within unit"
assert f"y_lag{SEASON}" in feat.columns and all(f"y_lag{SEASON - h}" in feat.columns for h in HORIZONS if SEASON - h > 0), "seasonal-naive lags missing"
assert len(feat) >= 30000, f"only {len(feat)} rows: collect more data (Step 1d) or lower FREQ"
print("Tests C: OK |", len(feat), "rows,", feat[UNIT_COL].nunique(), "stations")

Tests C: OK | 83722 rows, 37 stations


In [34]:
# Handover: parquet + manifest with md5
feat.to_parquet("data/processed/feat.parquet", index=False)
feat.to_csv("data/processed/feat.csv", index=False)                    # CSV copy of the handover table (parquet is the file of record)
md5 = hashlib.md5(open("data/processed/feat.parquet", "rb").read()).hexdigest()
manifest = {"group": GROUP, "topic": TOPIC, "rows": len(feat), "cols": list(feat.columns), "units": int(feat[UNIT_COL].nunique()), "start": str(feat[TIME_COL].min()), "end": str(feat[TIME_COL].max()),
            "freq": FREQ, "horizons": HORIZONS, "h_main": H_MAIN, "season": SEASON, "unit_col": UNIT_COL, "time_col": TIME_COL, "target_col": TARGET_COL,
            "exog_cols": [c for c in EXOG_COLS if c in feat.columns], "prim_covs": [c for c in PRIM_COVS if c in feat.columns],
            "event_abs": EVENT_ABS, "event_quantile": EVENT_QUANTILE, "test_start": TEST_START, "md5": md5}
json.dump(manifest, open("data/processed/manifest.json", "w"), indent=2); print(json.dumps(manifest, indent=2)[:900])

{
  "group": "Nhom 12",
  "topic": "Three-day river water temperature forecasting at continuous USGS stations from discharge and weather",
  "rows": 83722,
  "cols": [
    "station",
    "date",
    "wtemp",
    "TMAX",
    "PRCP",
    "dox",
    "turb",
    "flow",
    "dow",
    "mon",
    "doy",
    "doy_sin",
    "doy_cos",
    "y_lag1",
    "y_lag2",
    "y_lag3",
    "y_lag4",
    "y_lag6",
    "y_lag7",
    "y_lag14",
    "y_lag28",
    "y_ma_season",
    "y_sd_season",
    "y_diff1",
    "TMAX_roll3",
    "PRCP_roll3",
    "dox_roll3",
    "turb_roll3",
    "flow_roll3",
    "TMAX_lag1",
    "PRCP_lag1",
    "y_h1",
    "y_h3"
  ],
  "units": 37,
  "start": "2018-01-01 00:00:00",
  "end": "2026-09-14 00:00:00",
  "freq": "D",
  "horizons": [
    1,
    3
  ],
  "h_main": 3,
  "season": 7,
  "unit_col": "station",
  "time_col": "date",
  "target_col": "wtemp",
  "exog_cols": [
   


In [35]:
# Record the prompts you used this week in the AI Audit Log (one real row per prompt that changed your work; 3-5 per week)
# Uncomment and edit; do not log prompts you did not use. Example:
# audit("Step 2c", "Cột wtemp thiếu theo cụm tối đa 40 ngày ở một số trạm; nội suy hay bỏ?", "Claude", "khuyên interpolate(limit=3) rồi bỏ cụm dài", "đếm cụm > 3 ngày bằng groupby, đúng", "áp dụng limit=3", hallucination=False)
print(pd.read_csv(AUDIT_PATH).tail(3) if os.path.exists(AUDIT_PATH) else "audit log is empty: add your real prompts with audit(...)")

audit log is empty: add your real prompts with audit(...)


## Bài tập mở rộng cho Sinh viên A (làm ít nhất 3 trong 5, ghi kết quả vào report/exercises_C.md)

1. **Nguồn thứ ba**: thêm một nguồn phụ nữa (lịch ngày lễ, giá, sự kiện) bằng cùng hàm `load_table` và `merge_asof`; báo tỷ lệ dòng ghép được và tương quan với mục tiêu.
2. **So sánh tần suất**: gộp mục tiêu về tần suất thô hơn (ví dụ ngày) và so sức mạnh mùa giữa hai tần suất; kết luận tần suất nào phù hợp cho câu hỏi của đề.
3. **Hai truy vấn SQL thêm**: một truy vấn dùng `QUALIFY` để lấy top 5 đơn vị theo mỗi tháng, một truy vấn `WITH` tính tỷ lệ sự kiện theo mùa; giải thích kết quả bằng hai câu.
4. **Phân rã mùa**: dùng `statsmodels.tsa.seasonal.STL` trên một đơn vị; vẽ trend, seasonal, resid; nhận xét phần dư có còn cấu trúc không.
5. **Kiểm tra rò rỉ có chủ ý**: tạo một đặc trưng sai (dùng `LEAD`) rồi cho Sinh viên B chạy thử để thấy MAE giảm bất thường; ghi lại bài học vào Discussion.

In [36]:
# Exercise 4 starter: STL decomposition for the longest station (annual period on a daily grid)
from statsmodels.tsa.seasonal import STL
PERIOD = 365 if FREQ == "D" else SEASON
u0 = df.groupby(UNIT_COL).size().idxmax(); s = df_grid[df_grid[UNIT_COL] == u0].set_index(TIME_COL)[TARGET_COL].asfreq(FREQ).interpolate(limit=3).dropna()
if len(s) > 2 * PERIOD + 10:
    stl = STL(s, period=PERIOD, robust=True).fit()
    fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
    for ax, comp, lab in zip(axes, [stl.trend, stl.seasonal, stl.resid], ["trend", "seasonal", "residual"]): ax.plot(comp, color=TEAL, lw=0.8); ax.set_ylabel(lab); style(ax)
    savefig(fig, "fig_exercise_stl"); print("station", u0, "| residual share of variance:", round(float(stl.resid.var() / s.var()), 3))

saved report/fig_exercise_stl.png
station 01480617 | residual share of variance: 0.05


In [37]:
# Exercise 2 starter: seasonal strength by month at the daily versus the weekly frequency
weekly = df.set_index(TIME_COL).groupby(UNIT_COL)[TARGET_COL].resample("W").mean().reset_index()
strength = lambda frame, key: round(frame.groupby(key)[TARGET_COL].mean().std() / frame[TARGET_COL].std(), 3)
print("month-of-year strength: daily", strength(df.assign(k=df[TIME_COL].dt.month), "k"), "| weekly", strength(weekly.assign(k=weekly[TIME_COL].dt.month), "k"))

month-of-year strength: daily 0.974 | weekly 0.984


In [38]:
# Exercise 3 starter: QUALIFY and CTE examples (edit and add your own two queries to sql/queries.sql)
ex_sql = f"""
SELECT {UNIT_COL}, month({TIME_COL}) AS mon, AVG({TARGET_COL}) AS m, RANK() OVER (PARTITION BY month({TIME_COL}) ORDER BY AVG({TARGET_COL}) DESC) AS rk FROM clean GROUP BY 1, 2 QUALIFY rk <= 5 ORDER BY mon, rk;
"""
display(con.execute(ex_sql).df().head(10))

,station,mon,m,rk
0,01493112,1,6.583306,1
1,01576381,1,5.016129,2
2,01575598,1,4.765899,3
3,01650800,1,4.121505,4
4,01649500,1,3.929597,5
5,01493112,2,8.074107,1
6,01576381,2,6.135294,2
7,01575598,2,5.732828,3
8,01473500,2,5.715385,4
9,01650800,2,5.543307,5


### Tự đánh giá Notebook_C (điền trước khi gửi giảng viên)

| Tiêu chí | Tự chấm (0-2) | Bằng chứng |
|---|---|---|
| Nguồn và giấy phép rõ ràng | | report/data_sources.csv |
| So sánh hai nguồn có số | | table_source_comparison.csv |
| Làm sạch có nhật ký | | table_cleaning_log.csv |
| EDA có nhận xét kèm số | | eda_notes.md |
| SQL không rò rỉ, có kiểm tra | | features.sql, ô kiểm tra rò rỉ |
| Bài tập mở rộng | | exercises_C.md |
| AI Audit Log | | ai_audit_log.csv |

## Lỗi thường gặp và cách sửa

| Lỗi | Nguyên nhân | Cách sửa |
|---|---|---|
| `KeyError: 'ts'` | tên cột trong file khác CONFIG | in `df.columns`, sửa `TIME_COL` |
| Rò rỉ tương lai (điểm quá tốt) | dùng `shift(-k)` hoặc trung bình cả chuỗi khi tạo đặc trưng | chỉ dùng `LAG` và cửa sổ `PRECEDING`; kiểm tra `feat.ts` của đặc trưng luôn nhỏ hơn mục tiêu |
| Trùng dòng theo đơn vị và thời gian | nguồn có hai bản ghi cùng thời điểm | `drop_duplicates([UNIT_COL, TIME_COL])` rồi ghi vào nhật ký làm sạch |
| Lưới thời gian thiếu bước | không `resample(FREQ)` | resample và đếm bước thiếu theo đơn vị |
| Ghép nguồn phụ mất nửa dòng | lệch múi giờ hoặc lệch tần suất | đưa cả hai về UTC hoặc giờ địa phương thống nhất, dùng `merge_asof` với `tolerance` |
| DuckDB `Binder Error` | tên cột là từ khoá (`do`, `at`, `year`) hoặc trùng tên bảng | đổi tên cột, dùng dấu ngoặc kép |
| `MemoryError` | nạp toàn bộ file lớn bằng pandas | dùng `duckdb.read_csv_auto` và lọc sớm |
| Chronos không cài được | thiếu `pip install chronos-forecasting` | bỏ qua ô mô hình nền, ghi vào hạn chế |

## Checklist Notebook_C trước khi bàn giao (tick từng dòng)

- [ ] Ba file CSV: `data/raw/primary_snapshot.csv`, `data/processed/clean.csv`, `data/processed/feat.csv` mở được bằng Excel, số dòng khớp manifest
- [ ] `report/data_sources.csv` có đủ hai nguồn với URL, giấy phép, số dòng, khoảng thời gian
- [ ] `report/table_source_comparison.csv` và một câu nhận xét về mức nhất quán giữa hai nguồn
- [ ] `report/table_cleaning_log.csv` giải thích mọi dòng bị bỏ
- [ ] 9 hình EDA có caption nêu số liệu; `report/eda_notes.md` có 5 nhận xét kèm số
- [ ] `sql/queries.sql` 4 truy vấn RQ1 chạy qua; `report/table_q1..q4.csv`; `report/table_rq1_tests.csv` có p-value
- [ ] `sql/features.sql` không dùng LEAD ngoài mục tiêu; ô kiểm tra rò rỉ qua
- [ ] Test cases qua; `feat.parquet` và `manifest.json` gửi cho Sinh viên B
- [ ] AI Audit Log có ít nhất 6 mục cho tuần 1 và 2, ít nhất một mục `hallucination=True`

## Mẫu viết phần Data và RQ1 cho bài báo (điền số từ các bảng)

**Data.** We use {PRIMARY_SOURCE} ({license}, {rows} rows, {units} units, {start} to {end}, frequency {FREQ}) merged with {SECOND_SOURCE} ({share}% of rows have covariates). Cleaning removed {removed} rows (Table cleaning log); the two sources agree on {metric} (Table source comparison).

**RQ1.** The target varies most with {calendar or covariate}: seasonal strength {value} by {period} versus {value} by {period} (Figure EDA 4); units differ by a factor of {ratio} (Figure EDA 5); the covariate {name} leads the target by {lag} steps with correlation {rho} (Figure EDA 7); autocorrelation at the seasonal lag is {acf}, so the seasonal naive baseline is expected to be strong (Figure EDA 8). Kruskal-Wallis across units gives p = {p} (Table RQ1 tests).